# 大模型的调用完全指南

## 课程目标

通过本教程，你将学习：

1. **函数调用的基本原理**
2. **工具描述的格式与结构**
3. **基于 DeepSeek 的函数调用实现**
4. **函数调用与思考模式的结合**
5. **多轮对话中的函数调用**

每个章节都包含：概念讲解 + 代码示例 + 练习题 + 参考答案

---

## 一、函数调用简介

### 1.1 什么是函数调用

函数调用（Function Calling）是**让大模型决定何时调用外部工具/函数**的技术。

**核心价值**：让 AI 能够对接现实世界，获取实时信息、执行操作。

| 传统方式 | 函数调用方式 |
|----------|-------------|
| AI 只能回答训练数据中的知识 | AI 可以查询实时数据 |
| 无法执行实际操作 | 可以调用 API、数据库等 |
| 回答可能过时 | 获取最新信息 |

### 1.2 函数调用的工作流程

```
用户问题 → AI判断需要工具 → 返回工具名称和参数 → 执行工具函数 → 返回结果 → AI整合回答
```

### 1.3 完整流程图

1. **用户发送问题**
2. **AI 判断是否需要调用函数**
3. **返回函数调用请求**（包含函数名和参数）
4. **本地执行函数**
5. **将结果返回给 AI**
6. **AI 生成最终回答**

In [1]:
# 环境配置
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
from datetime import datetime

# 加载环境变量
load_dotenv()

# 初始化 DeepSeek 客户端
client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url='https://api.deepseek.com'
)

print('环境配置完成！')

环境配置完成！


---

## 二、工具描述的格式与结构

### 2.1 工具描述的完整结构

工具描述是一个 JSON 对象，告诉 AI 有哪些函数可用：

```python
tools = [
    {
        'type': 'function',
        'function': {
            'name': '函数名称',
            'description': '函数功能描述',
            'parameters': {
                'type': 'object',
                'properties': {
                    '参数名': {
                        'type': '参数类型',
                        'description': '参数说明'
                    }
                },
                'required': ['必填参数列表']
            }
        }
    }
]
```

### 2.2 参数类型映射

| Python 类型 | JSON Schema 类型 |
|-------------|------------------|
| str | string |
| int | integer |
| float | number |
| bool | boolean |
| list | array |
| dict | object |

In [2]:
# 定义天气查询工具
def get_weather(city):
    """通过城市名称获取天气信息"""
    weather_data = {
        '北京': '晴天，25℃',
        '上海': '多云，28℃',
        '广州': '小雨，30℃',
        '深圳': '阴天，27℃'
    }
    return f"{city} 当前天气：{weather_data.get(city, '未知')}"

# 工具描述
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': '获取指定城市的天气信息',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {
                        'type': 'string',
                        'description': '城市名称'
                    }
                },
                'required': ['city']
            }
        }
    }
]

print('工具定义完成！')

工具定义完成！


In [4]:
messages = [
        {'role': 'user', 'content': "广州天气怎样"}
    ]

# 2. 调用 API，传入工具描述
response = client.chat.completions.create(
        model='deepseek-chat',
        messages=messages,
        tools=tools,
        tool_choice='auto'
    )

# 3. 获取 AI 回复
message = response.choices[0].message

print("回复:",message)

回复: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_ad6eJH1knduRyriv7nWj8051', function=Function(arguments='{"city": "广州"}', name='get_weather'), type='function', index=0)])


In [5]:
response.choices[0].message.content

''

---

## 二. 工具描述自动生成

### 2.5.1 为什么需要自动生成

手动编写工具描述容易出错，且重复工作。使用工具描述生成函数可以：
- 自动解析函数名、文档字符串
- 自动识别参数类型和说明
- 自动生成符合 OpenAI 规范的工具描述

### 2.5.2 工具描述生成函数

我们先从 `tool_description_generator.py` 中导入生成函数，然后使用它来简化代码。

In [3]:
# 导入工具描述生成函数
import sys
sys.path.insert(0, 'E:\\2609\\智能体开发')
from tool_description_generator import generate_tool_description
import json

# 测试：用函数生成天气工具描述
print('=== 自动生成天气查询工具描述 ===')
weather_tool = generate_tool_description(get_weather)
print(json.dumps(weather_tool, ensure_ascii=False, indent=2))

=== 自动生成天气查询工具描述 ===
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "通过城市名称获取天气信息",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "null",
          "description": "city参数"
        }
      },
      "required": [
        "city"
      ]
    }
  }
}


现在我们更新工具定义，使用自动生成方式：

In [7]:
# 使用自动生成工具描述
weather_tool = generate_tool_description(get_weather)
tools = [weather_tool]

# 展示生成的工具描述
print('=== 自动生成的天气查询工具 ===')
print(json.dumps(tools, ensure_ascii=False, indent=2))

=== 自动生成的天气查询工具 ===
[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "通过城市名称获取天气信息",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "null",
            "description": "city参数"
          }
        },
        "required": [
          "city"
        ]
      }
    }
  }
]


In [8]:
messages = [
        {'role': 'user', 'content': "广州天气怎样"}
    ]

# 2. 调用 API，传入工具描述
response = client.chat.completions.create(
        model='deepseek-chat',
        messages=messages,
        tools=tools,
        tool_choice='auto'
    )

# 3. 获取 AI 回复
message = response.choices[0].message

print("回复:",message)

回复: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_epWE0SYC5gxpKANsn4Pr0668', function=Function(arguments='{"city": "广州"}', name='get_weather'), type='function', index=0)])


## 任务一 

完成如下工具的调用

In [4]:
def get_stock_price(symbol: str, date: str = None) -> float:
    """
    获取股票实时价格
        
    :param symbol: 股票代码，如AAPL、GOOGL
    :param date: 查询日期，格式YYYY-MM-DD，默认当前日期
    :return: 股票价格
    """
    return 175.50

In [6]:
stock_tool = generate_tool_description(get_stock_price)
stock_tool

{'type': 'function',
 'function': {'name': 'get_stock_price',
  'description': '获取股票实时价格',
  'parameters': {'type': 'object',
   'properties': {'symbol': {'type': 'string',
     'description': '股票代码，如AAPL、GOOGL'},
    'date': {'type': 'string', 'description': '查询日期，格式YYYY-MM-DD，默认当前日期'}},
   'required': ['symbol']}}}

In [7]:
messages = [
        {'role': 'user', 'content': "比亚迪股票2026年9月28日的价格"}
    ]

# 2. 调用 API，传入工具描述
response = client.chat.completions.create(
        model='deepseek-chat',
        messages=messages,
        tools=[stock_tool],
        tool_choice='auto'
    )

# 3. 获取 AI 回复
message = response.choices[0].message

print("回复:",message)

回复: ChatCompletionMessage(content="I'll look up BYD's (比亚迪) stock price for September 28, 2026.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_oGICEcfEuROkWAPZI82e2000', function=Function(arguments='{"symbol": "BYD", "date": "2026-09-28"}', name='get_stock_price'), type='function', index=0)])


In [13]:
message.tool_calls[0].function.arguments

'{"symbol": "BYD", "date": "2026-09-28"}'

In [10]:
message.tool_calls.function.name

AttributeError: 'list' object has no attribute 'function'

---

## 三、基于 DeepSeek 的函数调用实现

### 3.1 基本函数调用流程

使用 `tools` 参数注册工具，`tool_choice='auto'` 让 AI 自动决定是否调用工具。
现在我们使用**自动生成的工具描述**来简化代码：

In [12]:
# 完整的函数调用流程（使用自动生成工具描述）
def call_with_tools(user_question):
    # 1. 构建消息列表
    messages = [
        {'role': 'user', 'content': user_question}
    ]

    # 2. 调用 API，传入工具描述
    response = client.chat.completions.create(
        model='deepseek-chat',
        messages=messages,
        tools=tools,
        tool_choice='auto'
    )

    # 3. 获取 AI 回复
    message = response.choices[0].message

    print("第一次回复:",message)

    # 4. 检查是否有函数调用
    if message.tool_calls:
        print('需要调用函数！')
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            print(f'函数名：{func_name}')
            print(f'参数：{func_args}')

            # 5. 执行函数
            if func_name == 'get_weather':
                result = get_weather(**func_args)
                print(f'执行结果：{result}')

                # 6. 将结果返回给 AI
                messages.append({
                    'role': 'assistant',
                    'tool_calls': [
                        {
                            'id': tool_call.id,
                            'type': 'function',
                            'function': {
                                'name': func_name,
                                'arguments': tool_call.function.arguments
                            }
                        }
                    ]
                })
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tool_call.id,
                    'content': result
                })

                # 7. 再次调用获取最终回答
                final_response = client.chat.completions.create(
                    model='deepseek-chat',
                    messages=messages,
                    tools=tools,
                    tool_choice='auto'
                )

                return final_response.choices[0].message.content
    else:
        print('不需要调用函数，直接回答')
        return message.content

In [13]:
# 测试
result = call_with_tools('北京今天天气怎么样？')
print('\n最终回答：', result)

第一次回复: ChatCompletionMessage(content='好的，我来查询北京今天的天气情况。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_ytI8bcTIdtG9ZhG7Uvkn1944', function=Function(arguments='{"city": "北京"}', name='get_weather'), type='function', index=0)])
需要调用函数！
函数名：get_weather
参数：{'city': '北京'}
执行结果：北京 当前天气：晴天，25℃

最终回答： 北京今天天气是**晴天**，气温**25℃**，天气不错，适合出门活动哦！😊


### 任务二：添加股票查询函数

请添加一个股票查询函数，查询任意股票代码的股票价格。

要求：
1. 定义 get_stock_price 函数（带类型注解和完善的 docstring）
2. **使用 generate_tool_description 自动生成工具描述**
3. 测试查询股票价格

###  通过akshare获取股票价格

注意要先安装akshare

https://akshare.akfamily.xyz/data/stock/stock.html#id23

pip install akshare -i https://mirrors.aliyun.com/pypi/simple/

In [17]:
import akshare as ak
stock_data = ak.stock_zh_a_daily(symbol="sz300750", start_date="2025-04-03", end_date="2025-04-04", adjust="qfq")
stock_data

,date,open,high,low,close,volume,amount,outstanding_share,turnover
0,2025-04-03,234.79,237.37,232.44,232.95,23281639.0,5.693963e+09,3.902556e+09,0.005966


In [22]:
import akshare as ak

def get_stock_price(symbol: str):
    """获取股票当前价格
    
    :param symbol: 股票代码，如 sz000002、sz300750
    """
    stock_data = ak.stock_zh_a_daily(symbol=symbol, start_date="2025-04-03", end_date="2025-04-04", adjust="qfq")
    stock_data = stock_data.close.values[0]
    return f"{symbol} 当前价格：${stock_data}"

In [24]:
get_stock_price("sz002594")

'sz002594 当前价格：$117.33'

In [29]:
stock_tool = generate_tool_description(get_stock_price)
stock_tools = [stock_tool]

In [30]:
stock_tool

{'type': 'function',
 'function': {'name': 'get_stock_price',
  'description': '获取股票当前价格',
  'parameters': {'type': 'object',
   'properties': {'symbol': {'type': 'string',
     'description': '股票代码，如 sz000002、sz300750'}},
   'required': ['symbol']}}}

In [28]:
call_with_tools_stock('sz300750 股票价格是多少？')

第一次回复: ChatCompletionMessage(content="I'll look up the current stock price for you.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_fqxMNnUTPkabRoC0dBYN2310', function=Function(arguments='{"symbol": "sz300750"}', name='get_stock_price'), type='function', index=0)])
需要调用函数！
函数名：get_stock_price
参数：{'symbol': 'sz300750'}
执行结果：sz300750 当前价格：$232.95


'sz300750（宁德时代）当前价格为 **$232.95**。'

In [27]:
# 完整的函数调用流程（使用自动生成工具描述）
def call_with_tools_stock(user_question):
    # 1. 构建消息列表
    messages = [
        {'role': 'user', 'content': user_question}
    ]

    # 2. 调用 API，传入工具描述
    response = client.chat.completions.create(
        model='deepseek-chat',
        messages=messages,
        tools=stock_tools,
        tool_choice='auto'
    )

    # 3. 获取 AI 回复
    message = response.choices[0].message

    print("第一次回复:",message)

    # 4. 检查是否有函数调用
    if message.tool_calls:
        print('需要调用函数！')
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            print(f'函数名：{func_name}')
            print(f'参数：{func_args}')

            # 5. 执行函数
            if func_name == 'get_stock_price':
                result = get_stock_price(**func_args)
                print(f'执行结果：{result}')

                # 6. 将结果返回给 AI
                messages.append({
                    'role': 'assistant',
                    'tool_calls': [
                        {
                            'id': tool_call.id,
                            'type': 'function',
                            'function': {
                                'name': func_name,
                                'arguments': tool_call.function.arguments
                            }
                        }
                    ]
                })
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tool_call.id,
                    'content': result
                })

                # 7. 再次调用获取最终回答
                final_response = client.chat.completions.create(
                    model='deepseek-chat',
                    messages=messages,
                    tools=stock_tools,
                    tool_choice='auto'
                )

                return final_response.choices[0].message.content
    else:
        print('不需要调用函数，直接回答')
        return message.content

### 练习1参考答案

```python
# 1. 定义股票查询函数
def get_stock_price(symbol: str):
    """获取股票实时价格
    
    :param symbol: 股票代码，如 AAPL、GOOGL、MSFT
    """
    stock_data = {
        'AAPL': 175.50,
        'GOOGL': 141.80,
        'MSFT': 378.90
    }
    return f"{symbol} 当前价格：${stock_data.get(symbol, '未知')}"

# 2. 使用 generate_tool_description 自动生成工具描述
stock_tool = generate_tool_description(get_stock_price)
stock_tools = [stock_tool]

# 3. 测试
# 测试：'苹果公司的股票现在多少钱？'
```

---

## 四、函数调用与思考模式的结合

### 4.1 什么是思考模式

DeepSeek 的思考模式让模型在输出最终回答之前，先输出推理过程，提升复杂问题的准确率。

### 4.2 思考模式的参数

| 参数 | 说明 |
|------|------|
| reasoning_effort | 推理强度：high / max |
| extra_body={'thinking': {'type': 'enabled'}} | 启用思考模式 |
| response.choices[0].message.reasoning_content | 获取思考内容 |

### 4.3 结合使用的优势

- **复杂推理**：先思考是否需要调用函数
- **多步操作**：规划函数调用顺序
- **错误处理**：思考如何处理异常情况

**现在更新示例，使用自动生成工具描述：**

In [89]:
messages = [
        {'role': 'user', 'content': "中国的首都"}
    ]

# 2. 调用 API，传入工具描述
response = client.chat.completions.create(
        model='deepseek-chat',
        messages=messages,
        reasoning_effort='high',
        extra_body={'thinking': {'type': 'enabled'}}
    )

# 3. 获取 AI 回复
message = response.choices[0].message

print("回复:",message)

回复: ChatCompletionMessage(content='中国的首都是**北京**。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='我们需要回答用户中文问题“中国的首都”。需要直接回答。中国首都是北京。注意可能用户问“中国的首都”即可。应简洁。最终用中文回答。')


In [79]:
# 定义工具（带日期查询）并使用自动生成
def get_current_date() -> str:
    """获取当前日期
    
    :return: 返回格式为 YYYY-mm-dd 的日期字符串
    """
    return datetime.now().strftime('%Y-%m-%d')

def get_weather_with_date(city: str, date: str):
    """获取指定城市和日期的天气
    
    :param city: 城市名称，如 北京、上海、广州
    :param date: 查询日期，格式 YYYY-mm-dd
    """
    weather_data = {
        '北京': {'2026-10-11': '晴天，25℃', '2026-05-07': '多云，23℃'},
        '上海': {'2026-05-06': '小雨，20℃', '2026-05-07': '阴，22℃'},
    }
    return weather_data.get(city, {}).get(date, '暂无数据')

# 自动生成工具描述
date_tool = generate_tool_description(get_current_date)
weather_with_date_tool = generate_tool_description(get_weather_with_date)
thinking_tools = [date_tool, weather_with_date_tool]

tool_map = {
    'get_current_date': get_current_date,
    'get_weather_with_date': get_weather_with_date
}

print('=== 自动生成的工具描述 ===')
print(f'日期查询工具：\n{json.dumps(date_tool, ensure_ascii=False, indent=2)}')
print(f'\n天气查询工具：\n{json.dumps(weather_with_date_tool, ensure_ascii=False, indent=2)}')

=== 自动生成的工具描述 ===
日期查询工具：
{
  "type": "function",
  "function": {
    "name": "get_current_date",
    "description": "获取当前日期",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": []
    }
  }
}

天气查询工具：
{
  "type": "function",
  "function": {
    "name": "get_weather_with_date",
    "description": "获取指定城市和日期的天气",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "城市名称，如 北京、上海、广州"
        },
        "date": {
          "type": "string",
          "description": "查询日期，格式 YYYY-mm-dd"
        }
      },
      "required": [
        "city",
        "date"
      ]
    }
  }
}


In [82]:
# 带思考模式的函数调用（使用自动生成工具）
def chat_with_thinking_tools(user_input):
    messages = [{'role': 'user', 'content': user_input}]

    while True:
        # 调用 API：启用思考模式
        response = client.chat.completions.create(
            model='deepseek-chat',
            messages=messages,
            tools=thinking_tools,
            tool_choice='auto',
            reasoning_effort='high',
           extra_body={'thinking': {'type': 'enabled'}}
        )

        message = response.choices[0].message
        reasoning = message.reasoning_content

        # 打印思考过程
        if reasoning:
            print('\n【思考过程】')
            print(reasoning[:200] + '...' if len(reasoning) > 200 else reasoning)

        # 检查是否有函数调用
        if message.tool_calls:
            # 添加助手消息（包含思考内容和工具调用）
            assistant_msg = {
                'role': 'assistant',
                'content': message.content,
                'reasoning_content': reasoning,
                'tool_calls': [
                    {
                        'id': tc.id,
                        'type': 'function',
                        'function': {
                            'name': tc.function.name,
                            'arguments': tc.function.arguments
                        }
                    } for tc in message.tool_calls
                ]
            }
            messages.append(assistant_msg)

            # 执行工具函数
            for tc in message.tool_calls:
                func_name = tc.function.name
                func_args = json.loads(tc.function.arguments)
                result = tool_map[func_name](**func_args)
                print(f'\n【执行工具】{func_name}({func_args}) = {result}')

                # 添加工具结果消息
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tc.id,
                    'content': result
                })
        else:
            # 没有更多工具调用，返回最终回答
            print('\n【最终回答】')
            return message.content

In [83]:
# 测试
result = chat_with_thinking_tools('北京今天天气如何？')
print(result)


【思考过程】
The user asks about Beijing's weather today. I need to get the current date first, then get the weather. Actually, get_weather_with_date requires a date. Let me get current date first.

I could call b...

【执行工具】get_current_date({}) = 2026-10-11

【思考过程】
Now get the weather for Beijing on 2026-10-11.

【执行工具】get_weather_with_date({'city': '北京', 'date': '2026-10-11'}) = 晴天，25℃

【最终回答】
北京今天（2026年10月11日）天气晴朗，气温约 25℃，体感比较舒适，适合外出活动。


## 任务

1 通过获取日期与股票查询联合，获得某一天的股票价格

In [87]:
stock_zh_a_daily_qfq_df = ak.stock_zh_a_daily(symbol="sz300750", start_date="2026-10-09", end_date="2026-10-09", adjust="qfq")
print(stock_zh_a_daily_qfq_df)

         date   open    high    low   close      volume        amount  \
0  2026-10-09  292.0  301.42  289.0  297.75  54152671.0  1.602812e+10   

   outstanding_share  turnover  
0       4.260617e+09   0.01271  


In [85]:
def get_stock_price_date(symbol: str, date: str):
    """获取股票某一天的股票价格
    
    :param symbol: 股票代码，如 sz000002、sz300750
    :param date: 日期，如 2025-04-03
    :return: 股票价格
    """
    stock_data = ak.stock_zh_a_daily(symbol=symbol, start_date=date, end_date=date, adjust="qfq")
    print(stock_data)
    stock_data = stock_data.close.values[0]
    return f"{symbol} {date} 价格：${stock_data}"

# 定义工具（带日期查询）并使用自动生成
def get_current_date() -> str:
    """获取当前日期
    
    :return: 返回格式为 YYYY-mm-dd 的日期字符串
    """
    return datetime.now().strftime('%Y-%m-%d')

# 自动生成工具描述
date_tool = generate_tool_description(get_current_date)
stock_price_date_tool = generate_tool_description(get_stock_price_date)
thinking_tools = [date_tool, stock_price_date_tool]

tool_map = {
    'get_current_date': get_current_date,
    'get_stock_price_date': get_stock_price_date
}


In [73]:
get_stock_price_date("sz000002", "2026-10-09")

         date  open  high   low  close       volume        amount  \
0  2026-10-09  4.13  4.14  3.89   4.06  589754855.0  2.355335e+09   

   outstanding_share  turnover  
0       9.715113e+09  0.060705  


'sz000002 2026-10-09 价格：$4.06'

In [74]:
# 测试
result = chat_with_thinking_tools('股票sz300750 2天前的股票价格是多少？')
print(result)



【思考过程】
The user asks for stock price of sz300750 2 days ago. I need current date first.

【执行工具】get_current_date({}) = 2026-10-11

【思考过程】
Current date is 2026-10-11. Two days ago = 2026-10-09. Let me get the price.
         date   open    high    low   close      volume        amount  \
0  2026-10-09  292.0  301.42  289.0  297.75  54152671.0  1.602812e+10   

   outstanding_share  turnover  
0       4.260617e+09   0.01271  

【执行工具】get_stock_price_date({'symbol': 'sz300750', 'date': '2026-10-09'}) = sz300750 2026-10-09 价格：$297.75

【最终回答】
当前日期是 **2026-10-11**，2 天前为 **2026-10-09**。

股票 **sz300750**（宁德时代）在 2026-10-09 的价格为 **$297.75**。


### 练习2：设计一个计算器工具

请设计一个支持加减乘除的计算器工具，并结合思考模式进行测试。

要求：
1. 定义 calculate 函数（带类型注解和完善的 docstring）
2. **使用 generate_tool_description 自动生成工具描述**
3. 测试复杂表达式：'计算 (25 + 10) * 2 除以 5 等于多少？'

### 练习2参考答案

```python
import math

def calculate(expression: str) -> str:
    """执行数学计算
    
    :param expression: 数学表达式，如 25 + 10，支持 +-*/() 运算符
    """
    try:
        # 安全计算，只允许数字和运算符
        allowed_chars = set('0123456789+-*/(). ')
        if all(c in allowed_chars for c in expression):
            result = eval(expression)
            return f"计算结果：{expression} = {result}"
        else:
            return '只支持数字和 +-*/() 运算符'
    except Exception as e:
        return f'计算错误：{e}'

# 使用 generate_tool_description 自动生成工具描述
calculate_tool = generate_tool_description(calculate)
calculate_tools = [calculate_tool]

# 测试：'计算 (25 + 10) * 2 除以 5 等于多少？'
```

In [1]:
import akshare as ak

stock_news_em_df = ak.stock_news_em(symbol="603777")
print(stock_news_em_df)

      关键词                                  新闻标题  \
0  603777               主动调整蓄力！603777，营收增长，净利承压   
1  603777  来伊份603777.SH)2025年净利润为-1.61亿元，同比亏损放大   
2  603777              来伊份一季度亏损1404万元，同比降212.9%   
3  603777                   来伊份2025年亏损扩大至1.61亿元   
4  603777                        涨停股复盘：29股封单超亿元   
5  603777            来伊份：2026年一季度实现营业总收入10.36亿元   
6  603777               亏损扩大、毛利率连降，来伊份上市后最大危机来袭   
7  603777       来伊份毛利率连降三年、连亏两年，引入“半导体资本大佬”能否救场   
8  603777                    来伊份年亏超2亿！直营收缩，加盟放量   
9  603777      来伊份：2025年营收38.19亿元 加盟业务增长105.08%   

                                                新闻内容                 发布时间  \
0  【导读】来伊份2025年业绩出炉 中国基金报记者郑俊婷 4月28日晚间，老牌零食品牌商来伊份...  2026-04-29 10:57:00   
1  2026年4月29日，来伊份603777.SH)发布2025年年报。 公司营业总收入为38....  2026-04-29 12:03:28   
2  4月28日，来伊份603777公布2026年一季报，公司营业收入为10.4亿元，同比下降1....  2026-04-29 11:23:00   
3  4月28日，来伊份603777公布2025年年报，公司营业收入为38.2亿元，同比上升13....  2026-04-29 11:39:00   
4  4140.80 有色金属 000863 三湘印象 5.34 4.21 758.69 4051... 

In [4]:
stock_news_em_df["新闻内容"][0]

'【导读】来伊份2025年业绩出炉 中国基金报记者郑俊婷 4月28日晚间，老牌零食品牌商来伊份（证券代码：603777）公布2025年年报。 受行业存量竞争及公司主动调整影响，来伊份2025年净利润大幅下滑。'

In [5]:
import akshare as ak

stock_financial_report_sina_df = ak.stock_financial_report_sina(stock="sh600600", symbol="资产负债表")
print(stock_financial_report_sina_df)

          报告日 流动资产           货币资金 结算备付金 拆出资金       交易性金融资产 买入返售金融资产 衍生金融资产  \
0    20260331  NaN  13254301120.0   NaN  NaN  9711512782.0      NaN    NaN   
1    20251231  NaN  12859566861.0   NaN  NaN  6228400545.0      NaN    NaN   
2    20250930  NaN  13979812320.0   NaN  NaN  7115820512.0      NaN    NaN   
3    20250630  NaN  16842451076.0   NaN  NaN  5179128769.0      NaN    NaN   
4    20250331  NaN  16295533617.0   NaN  NaN  4951523501.0      NaN    NaN   
..        ...  ...            ...   ...  ...           ...      ...    ...   
111  19941231  NaN    712428000.0   NaN  NaN     1055000.0      NaN    NaN   
112  19940630  NaN            NaN   NaN  NaN           NaN      NaN    NaN   
113  19931231  NaN    721310202.0   NaN  NaN   174645624.0      NaN    NaN   
114  19930630  NaN    107312388.0   NaN  NaN     2459000.0      NaN    NaN   
115  19921231  NaN    126113031.0   NaN  NaN           NaN      NaN    NaN   

       应收票据及应收账款       应收票据  ...   归属于母公司股东权益合计       少数股东权益 所有

（注意：多轮对话示例同样可以使用 generate_tool_description 来简化工具描述，
方法与前面示例相同。）

---

## 七、总结

### 核心要点

| 概念 | 说明 |
|------|------|
| 工具描述 | JSON 格式，告诉 AI 有哪些函数可用 |
| 工具调用 | AI 决定调用函数并返回参数 |
| 执行函数 | 本地执行函数，获取结果 |
| 思考模式 | 先推理再回答，提升准确性 |
| 多轮调用 | 支持复杂的函数调用序列 |
| **工具自动生成** | 使用 generate_tool_description 函数简化工具描述编写 |

### 工具自动生成优势

1. **减少重复劳动**：不用手动写冗长的 JSON
2. **减少错误**：自动生成格式规范
3. **可维护性好**：更新函数时自动更新工具描述
4. **代码更清晰**：只关注业务逻辑

### 使用方法

```python
# 1. 函数要带类型注解和完善的 docstring
def get_weather(city: str):
    """获取天气信息
    
    :param city: 城市名称
    """
    return ...

# 2. 使用 generate_tool_description 自动生成
from tool_description_generator import generate_tool_description
tool = generate_tool_description(get_weather)
tools = [tool]
```